In [ ]:
from minio import Minio
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col, when, length 
from pyspark.sql.types import NumericType, IntegerType, LongType, FloatType, DoubleType, DecimalType, DateType, TimestampType
from pyspark.sql import Window
import pyspark.sql.functions as F
from dotenv import load_dotenv, find_dotenv
import psycopg2
load_dotenv(find_dotenv())
import os

In [ ]:
JAR_PATH_1 = os.path.abspath("./jars/hadoop-aws-3.4.0.jar")
JAR_PATH_2 = os.path.abspath("./jars/aws-sdk-s3-2.29.52.jar")

JARS_LIST = f"{JAR_PATH_1},{JAR_PATH_2}"

In [ ]:
spark = (
    SparkSession.builder.appName("analysis")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.jars.repositories", "https://repo1.maven.org/maven2/")
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT"))
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY"))
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

In [ ]:
conn = psycopg2.connect(
    host="localhost",
    database=os.getenv("POSTGRES_DATABASE_NAME"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
)


LOAD DATA FROM POSTGRE DATABASE REMAINING

# Customer Related Analysis 

Adding Date Column

In [ ]:
dataframes["customers"] = dataframes["customers"].withColumn(
    "account_created_date",
    F.to_date("account_created_at")
)

Time Grain function

In [ ]:
def add_time_grain(df, date_col="account_created_date", grain="day"):
    if grain == "day":
        return df.withColumn("grain_date", F.col(date_col))
    elif grain == "week":
        return df.withColumn("grain_year", F.year(date_col)) \
                 .withColumn("grain_week", F.weekofyear(date_col))
    elif grain == "month":
        return df.withColumn("grain_year", F.year(date_col)) \
                 .withColumn("grain_month", F.month(date_col))
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

Analysis: Net Revenue vs. Net Profit

In [ ]:
def analyze_financial_health(orders_df, date_col="order_placed_at", grain="month"):
    df_g = add_time_grain(orders_df, date_col, grain)
    
    if grain == "day":
        group_cols = ["grain_date"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week"]
    else: # month
        group_cols = ["grain_year", "grain_month"]

    financial_df = (
        df_g.groupBy(*group_cols)
            .agg(
                F.sum("net_revenue").alias("total_net_revenue"),
                F.sum("net_profit").alias("total_net_profit"),
                F.count("order_id").alias("total_orders")
            )
            .withColumn("period_margin_pct", 
                        F.round((F.col("total_net_profit") / F.col("total_net_revenue")) * 100, 2))
            .orderBy(*group_cols)
    )
    
    return financial_df

aily_health = analyze_financial_health(dataframes["orders"], grain="day")

monthly_health = analyze_financial_health(dataframes["orders"], grain="month")


Analysis: Margin by Category (The "Drag" Analysis)

In [ ]:
def analyze_category_margins(products_df):
   
    category_df = (
        products_df.groupBy("category")
            .agg(
                F.avg("profit_margin").alias("avg_profit_margin"),
                F.sum("total_profit").alias("total_category_profit"),
                F.sum("total_revenue").alias("total_category_revenue"),
                F.sum("total_units_sold").alias("units_sold")
            )
            .orderBy(F.col("avg_profit_margin").asc())
    )
    
    return category_df


low_margin_cats = analyze_category_margins(dataframes["products"])

active customers over time

In [ ]:

def active_customers_over_time(df, grain="day"):
    if grain == "day":
        group_cols = [F.col("account_created_date").alias("date")]
    elif grain == "week":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.weekofyear("account_created_date").alias("week")
        ]
    elif grain == "month":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.month("account_created_date").alias("month")
        ]
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

    return (
        df.filter(F.col("is_active") == True)
          .groupBy(*group_cols)
          .agg(F.countDistinct("customer_id").alias("active_customers"))
          .orderBy(*group_cols)
    )

daily_active   = active_customers_over_time(dataframes["customers"], "day")
weekly_active  = active_customers_over_time(dataframes["customers"], "week")
monthly_active = active_customers_over_time(dataframes["customers"], "month")

account_status over time 

In [ ]:
def status_distribution_over_time(df, grain="day"):
    if grain == "day":
        group_cols = [F.col("account_created_date").alias("date")]
    elif grain == "week":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.weekofyear("account_created_date").alias("week")
        ]
    elif grain == "month":
        group_cols = [
            F.year("account_created_date").alias("year"),
            F.month("account_created_date").alias("month")
        ]
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

    return (
        df.groupBy(*(group_cols + [F.col("account_status")]))
          .agg(F.countDistinct("customer_id").alias("customer_count"))
          .orderBy(*group_cols, "account_status")
    )

daily_status   = status_distribution_over_time(dataframes["customers"], "day")
monthly_status = status_distribution_over_time(dataframes["customers"], "month")

New customers per day/week/month

In [ ]:
def new_customers(df, grain="day"):
    df_g = add_time_grain(df, grain=grain)

    if grain == "day":
        group_cols = ["grain_date"]
        order_cols = ["grain_date"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week"]
        order_cols = ["grain_year", "grain_week"]
    else:   # month
        group_cols = ["grain_year", "grain_month"]
        order_cols = ["grain_year", "grain_month"]

    new_df = (
        df_g.groupBy(*group_cols)
            .agg(F.countDistinct("customer_id").alias("new_customers"))
            .orderBy(*order_cols)
    )
    return new_df

daily_new   = new_customers(dataframes["customers"], "day")
weekly_new  = new_customers(dataframes["customers"], "week")
monthly_new = new_customers(dataframes["customers"], "month")

Cumulative customer growth curve

In [ ]:
def cumulative_customers(df, grain="day"):
    new_df = new_customers(df, grain)

    # Define window by time order
    if grain == "day":
        window = Window.orderBy("grain_date") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    elif grain == "week":
        window = Window.orderBy("grain_year", "grain_week") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    else:   # month
        window = Window.orderBy("grain_year", "grain_month") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)

    cum_df = new_df.withColumn(
        "cumulative_customers",
        F.sum("new_customers").over(window)
    )
    return cum_df

daily_growth   = cumulative_customers(dataframes["customers"], "day")
weekly_growth  = cumulative_customers(dataframes["customers"], "week")
monthly_growth = cumulative_customers(dataframes["customers"], "month")

Total new customers by geography + time

In [ ]:
geo_acquisition = (
    dataframes["customers"]
    .groupBy("country", "state_province", "city")
    .agg(F.countDistinct("customer_id").alias("new_customers"))
)

def geo_acquisition_over_time(df, grain="day"):
    df_g = add_time_grain(df, grain=grain)

    if grain == "day":
        group_cols = ["grain_date", "country", "state_province", "city"]
        order_cols = ["grain_date", "country", "state_province", "city"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week", "country", "state_province", "city"]
        order_cols = ["grain_year", "grain_week", "country", "state_province", "city"]
    else:  # month
        group_cols = ["grain_year", "grain_month", "country", "state_province", "city"]
        order_cols = ["grain_year", "grain_month", "country", "state_province", "city"]

    return (
        df_g.groupBy(*group_cols)
            .agg(F.countDistinct("customer_id").alias("new_customers"))
            .orderBy(*order_cols)
    )

daily_geo   = geo_acquisition_over_time(dataframes["customers"], "day")
monthly_geo = geo_acquisition_over_time(dataframes["customers"], "month")

Customer distribution by age group, city, state, country

In [ ]:
age_group_dist = (
    dataframes["customers"]
    .groupBy("customer_age_group")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("customer_age_group")
)
city_dist = (
    dataframes["customers"]
    .groupBy("country", "state_province", "city")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("country", "state_province", "city")
)
state_dist = (
    dataframes["customers"]
    .groupBy("country", "state_province")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("country", "state_province")
)
country_dist = (
    dataframes["customers"]
    .groupBy("country")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("country")
)

# Age group distribution and spending patterns

In [ ]:
age_group_spending = (
    dataframes["customers"]
    .groupBy("customer_age_group")
    .agg(
        F.countDistinct("customer_id").alias("customer_count"),
        F.avg("order_total_spent").alias("avg_order_total_spent"),
        F.avg("customer_lifetime_value").alias("avg_clv"),
        F.sum("order_total_spent").alias("total_spent"),
        F.sum("total_revenue").alias("total_revenue_age_group")
    )
    .orderBy("customer_age_group")
)

Gender-based product preferences

In [ ]:
cust_orders = (
    dataframes["orders"]
    .select("order_id", "customer_id")
    .join(
        dataframes["customers"].select("customer_id", "gender"),
        on="customer_id",
        how="inner"
    )
)

cust_order_items = (
    cust_orders
    .join(dataframes["order_items"].select("order_id", "product_id", "quantity"), on="order_id", how="inner")
    .join(dataframes["products"].select("product_id", "product_name", "category", "sub_category", "brand"),
          on="product_id",
          how="left")
)

# Gender-based preferences by category
gender_category_pref = (
    cust_order_items
    .groupBy("gender", "category")
    .agg(
        F.sum("quantity").alias("total_units"),
        F.countDistinct("product_id").alias("distinct_products"),
        F.countDistinct("order_id").alias("orders_count")
    )
    .orderBy("gender", F.col("total_units").desc())
)

# Optional: gender-based preferences by product (top-selling per gender)
gender_product_pref = (
    cust_order_items
    .groupBy("gender", "product_id", "product_name", "category")
    .agg(
        F.sum("quantity").alias("total_units"),
        F.countDistinct("order_id").alias("orders_count")
    )
    .orderBy("gender", F.col("total_units").desc())
)

New vs returning customers

In [ ]:
dataframes["customers"] = dataframes["customers"].withColumn(
    "customer_type",
    F.when(F.col("is_repeat_customer") == 1, F.lit("returning"))
     .otherwise(F.lit("new"))
)
new_vs_returning_country = (
    dataframes["customers"]
    .groupBy("country", "customer_type")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("country", "customer_type")
)
new_vs_returning_city = (
    dataframes["customers"]
    .groupBy("country", "state_province", "city", "customer_type")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("country", "state_province", "city", "customer_type")
)

new_vs_returning_state = (
    dataframes["customers"]
    .groupBy("country", "state_province", "customer_type")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("country", "state_province", "customer_type")
)

Total & average engagement per customer

In [ ]:
engagement_per_customer = dataframes["customers"].select(
    "customer_id",
    "total_sessions",
    "total_pages_viewed",
    "total_products_viewed"
)

engagement_overall = dataframes["customers"].agg(
    F.sum("total_sessions").alias("total_sessions_all_customers"),
    F.avg("total_sessions").alias("avg_sessions_per_customer"),
    F.sum("total_pages_viewed").alias("total_pages_viewed_all_customers"),
    F.avg("total_pages_viewed").alias("avg_pages_viewed_per_customer"),
    F.sum("total_products_viewed").alias("total_products_viewed_all_customers"),
    F.avg("total_products_viewed").alias("avg_products_viewed_per_customer")
)

Session-to-order behavior

In [ ]:
avg_session_to_order = dataframes["customers"].agg(
    F.avg("session_conversion_rate").alias("avg_session_conversion_rate"),
    F.avg("cart_abandonment_rate").alias("avg_cart_abandonment_rate")
)

distribution percentage for conversion & abandonment

In [ ]:
conv_percentage = dataframes["customers"].withColumn(
    "session_conversion_percentage",
    F.when(F.col("session_conversion_rate") < 0.1, "<10%")
     .when(F.col("session_conversion_rate") < 0.25, "10–25%")
     .when(F.col("session_conversion_rate") < 0.5, "25–50%")
     .when(F.col("session_conversion_rate") < 0.75, "50–75%")
     .otherwise("75%+")
)

conv_dist = (
    conv_percentage
    .groupBy("session_conversion_percentage")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("session_conversion_percentage")
)

abandon_percentage = dataframes["customers"].withColumn(
    "cart_abandonment_percentage",
    F.when(F.col("cart_abandonment_rate") < 0.1, "<10%")
     .when(F.col("cart_abandonment_rate") < 0.25, "10–25%")
     .when(F.col("cart_abandonment_rate") < 0.5, "25–50%")
     .when(F.col("cart_abandonment_rate") < 0.75, "50–75%")
     .otherwise("75%+")
)

abandon_dist = (
    abandon_percentage
    .groupBy("cart_abandonment_percentage")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .orderBy("cart_abandonment_percentage")
)

Basic correlation: tenure vs. spending

In [ ]:
corr_tenure_spend = dataframes["customers"].select(
    F.corr("customer_tenure_days", "order_total_spent").alias("corr_tenure_order_total_spent"),
    F.corr("customer_tenure_days", "customer_lifetime_value").alias("corr_tenure_clv")
)


Tenure buckets vs. average spending

In [ ]:
tenure_buckets_df = dataframes["customers"].withColumn(
    "tenure_bucket",
    F.when(F.col("customer_tenure_days") < 30, "<30d")
     .when(F.col("customer_tenure_days") < 90, "30–89d")
     .when(F.col("customer_tenure_days") < 180, "90–179d")
     .when(F.col("customer_tenure_days") < 365, "180–364d")
     .when(F.col("customer_tenure_days") < 730, "1–2y")
     .otherwise("2y+")
)

tenure_spend_stats = (
    tenure_buckets_df
    .groupBy("tenure_bucket")
    .agg(
        F.countDistinct("customer_id").alias("customer_count"),
        F.avg("order_total_spent").alias("avg_order_total_spent"),
        F.avg("customer_lifetime_value").alias("avg_clv"),
        F.sum("order_total_spent").alias("total_spent")
    )
    .orderBy("tenure_bucket")
)


# Product/category viewing patterns

Category-level viewing effectiveness

In [ ]:
category_view_patterns = (
    dataframes["products"]
    .groupBy("category")
    .agg(
        F.countDistinct("product_id").alias("products_in_category"),
        F.sum("total_units_sold").alias("total_units_sold"),
        F.sum("total_orders").alias("total_orders"),
        F.avg("view_to_purchase_rate").alias("avg_view_to_purchase_rate"),
        F.avg("revenue_per_view").alias("avg_revenue_per_view"),
        F.sum("total_revenue").alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc_nulls_last())
)

Product-level Top-View-to-Purchase Rates

In [ ]:
top_view_to_purchase_products = (
    dataframes["products"]
    .select(
        "product_id",
        "product_name",
        "category",
        "view_to_purchase_rate",
        "revenue_per_view",
        "total_units_sold",
        "total_orders"
    )
    .orderBy(F.col("view_to_purchase_rate").desc_nulls_last())
)

# Wishlist usage and conversion rate

Overall wishlist usage and conversion

In [ ]:
wishlist_overall = dataframes["wishlist"].agg(
    F.count("*").alias("total_wishlist_items"),
    F.countDistinct("customer_id").alias("customers_using_wishlist"),
    F.countDistinct("product_id").alias("products_in_wishlist"),
    F.count("*").filter(F.col("purchased_date").isNotNull()).alias("wishlist_purchased_items")
).withColumn(
    "wishlist_conversion_rate",
    F.col("wishlist_purchased_items") / F.col("total_wishlist_items")
)

Wishlist usage & conversion by product

In [ ]:
wishlist_by_product = (
    dataframes["wishlist"]  
    .groupBy("product_id")
    .agg(
        F.count("*").alias("wishlist_adds"),
        F.count("*").filter(F.col("purchased_date").isNotNull()).alias("wishlist_purchases")
    )
    .withColumn(
        "wishlist_conversion_rate",
        F.col("wishlist_purchases") / F.col("wishlist_adds")
    )
)

Wishlist usage & conversion by customer

In [ ]:
wishlist_by_customer = (
    dataframes["wishlist"]  
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("wishlist_adds"),
        F.count("*").filter(F.col("purchased_date").isNotNull()).alias("wishlist_purchases")
    )
    .withColumn(
        "wishlist_conversion_rate",
        F.when(F.col("wishlist_adds") > 0,
               F.col("wishlist_purchases") / F.col("wishlist_adds"))
         .otherwise(F.lit(0.0))
    )
)

Cart creation, abandonment, and recovery statistics

Basic cart creation & status distribution

In [ ]:
cart_stats_overall = dataframes["carts"].agg(
    F.countDistinct("cart_id").alias("total_carts"),
    F.count("*").alias("total_cart_lines")
)

cart_status_dist = (
    dataframes["carts"]
    .groupBy("cart_status")
    .agg(
        F.countDistinct("cart_id").alias("carts_count"),
        F.count("*").alias("cart_lines_count")
    )
    .orderBy("cart_status")
)

Abandonment & recovery (using agg_cart_abandonment_analysis)

In [ ]:

cart_abandon_overall = dataframes["cart_abandon"].agg(
    F.countDistinct("cart_id").alias("total_carts_tracked"),
    F.countDistinct("cart_id").filter(F.col("cart_status") == "abandoned").alias("abandoned_carts"),
    F.countDistinct("cart_id").filter(F.col("cart_status") == "purchased").alias("purchased_carts")
).withColumn(
    "abandonment_rate",
    F.col("abandoned_carts") / F.col("total_carts_tracked")
).withColumn(
    "purchase_rate",
    F.col("purchased_carts") / F.col("total_carts_tracked")
)

Value and size characteristics of abandoned vs purchased carts

In [ ]:
cart_value_stats = (
    dataframes["cart_abandon"]
    .groupBy("cart_status")
    .agg(
        F.countDistinct("cart_id").alias("carts_count"),
        F.avg("cart_total_value").alias("avg_cart_value"),
        F.avg("cart_items_count").alias("avg_cart_items"),
        F.avg("time_in_cart_days").alias("avg_time_in_cart_days"),
        F.avg("recovery_potential_score").alias("avg_recovery_potential_score")
    )
    .orderBy("cart_status")
)

Recovery opportunity: high-value abandoned carts

In [ ]:
high_value_abandoned = (
    dataframes["cart_abandon"]
    .filter(
        (F.col("cart_status") == "abandoned") &
        (F.col("cart_total_value") >= 100)  # threshold – adjust as needed
    )
    .select(
        "cart_id",
        "customer_id",
        "cart_total_value",
        "cart_items_count",
        "time_in_cart_days",
        "recovery_potential_score",
        "abandonment_risk_score"
    )
)